In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}

# fabric-rlm 0.1.10 adaptive smoke
Single-call sanity check: install wheel from OneLake, instantiate `FabricLM`, run one `engine='adaptive'` query, write a JSON summary back to the Lakehouse so the controller can read it.

In [ ]:
import sys, json, time, traceback, uuid, platform as _platform
from pathlib import Path

RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
FILES_ROOT = Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_smoke' / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

summary = {
    'run_id': RUN_ID,
    'started_at': time.time(),
    'python': _platform.python_version(),
    'stages': [],
    'passed': False,
    'error': None,
}

SUMMARY_PATH = RUN_ROOT / 'summary.json'

def write_summary():
    summary['updated_at'] = time.time()
    summary['elapsed_seconds'] = summary['updated_at'] - summary['started_at']
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

def stage(name, **fields):
    summary['stages'].append({'stage': name, 't': time.time(), **fields})
    print(f'[stage] {name}', fields if fields else '')
    write_summary()

stage('setup', run_root=str(RUN_ROOT))

In [ ]:
WHEEL_PATH = '/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.10-py3-none-any.whl'
stage('wheel_check', exists=Path(WHEEL_PATH).exists(), size=Path(WHEEL_PATH).stat().st_size if Path(WHEEL_PATH).exists() else 0)
import subprocess
out = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--force-reinstall', '--no-deps', WHEEL_PATH], capture_output=True, text=True)
stage('pip_wheel', rc=out.returncode, stderr_tail=out.stderr[-400:])
if out.returncode != 0:
    summary['error'] = 'wheel install failed'; write_summary(); raise SystemExit('wheel install failed')

try:
    import dspy
    stage('dspy_present', version=getattr(dspy,'__version__','?'))
except ImportError:
    out2 = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'dspy>=3.1.2'], capture_output=True, text=True)
    stage('pip_dspy', rc=out2.returncode, stderr_tail=out2.stderr[-400:])
    if out2.returncode != 0:
        summary['error'] = 'dspy install failed'; write_summary(); raise SystemExit('dspy install failed')
    import dspy
    stage('dspy_installed', version=getattr(dspy,'__version__','?'))

for mod in [m for m in list(sys.modules) if m == 'fabric_rlm' or m.startswith('fabric_rlm.')]:
    sys.modules.pop(mod, None)
import fabric_rlm
stage('imported', version=getattr(fabric_rlm, '__version__', '?'))

In [ ]:
from fabric_rlm import RLM, FabricLM

try:
    cheap = FabricLM('gpt-4.1-mini', temperature=0.0, cache=False)
    strong = FabricLM('gpt-5', reasoning_effort='medium', cache=False)
    stage('lms_built', cheap='gpt-4.1-mini', strong='gpt-5')
except Exception as exc:
    summary['error'] = 'FabricLM build failed: ' + repr(exc)
    summary['traceback'] = traceback.format_exc()
    write_summary()
    raise

def validator(result):
    payload = result.payload or {}
    ans = (payload.get('answer') or '').strip().lower()
    return '4' in ans

rlm = RLM(
    signature='question -> answer',
    lm=cheap,
    engine='adaptive',
    adaptive=dict(
        strong_lm=strong,
        validator=validator,
        max_attempts=2,
        parallel_rollouts=1,
    ),
)
stage('rlm_built')

try:
    t0 = time.perf_counter()
    result = rlm.run({'question': 'What is 2 + 2? Answer with just the number.'})
    elapsed = time.perf_counter() - t0
    meta = (result.trajectory.metadata or {}).get('adaptive', {}) if result.trajectory else {}
    summary['result'] = {
        'answer': (result.payload or {}).get('answer'),
        'submitted': result.submitted,
        'failure_reason': result.failure_reason,
        'elapsed_seconds': elapsed,
        'winner_rung': meta.get('winner_rung'),
        'attempts': [{'rung': a.get('rung'), 'passed': a.get('passed')} for a in meta.get('attempts', [])],
        'stop_reason': meta.get('stop_reason'),
    }
    summary['passed'] = bool(result.submitted) and bool(summary['result']['answer'])
    stage('run_complete', **summary['result'])
except Exception as exc:
    summary['error'] = repr(exc)
    summary['traceback'] = traceback.format_exc()
    stage('run_failed', error=repr(exc))

write_summary()
print('PASSED=', summary['passed'])